# 10 - Foundry IQ: Web IQ + Content Understanding Blob chunks

## Required permissions

| Identity | Permission | Function |
|---|---|---|
| Signed-in user | **Search Service Contributor** on Search | Author the two sources and independent knowledge base |
| Signed-in user | **Search Index Data Reader** (or **Search Index Data Contributor**) on Search | Inspect and retrieve existing Blob chunks |
| Search managed identity | **Cognitive Services OpenAI User** on the Foundry account | Keyless chat planning/synthesis and existing index vectorizer inference |
| Web IQ profile manager | Authority to issue an authorized profile key | Supply `WEBIQ_API_KEY` for Search's stored-header Web IQ calls |

**Resource prefix:** set `WORKSHOP_PREFIX` in your `.env` (3-20 lowercase letters/digits, starting with a letter).
Prefixes prevent naming collisions, not enforce access isolation. This lab creates only
`<prefix>-nb10-webiq`, `<prefix>-nb10-blob` and `<prefix>-nb10-kb`.

Progress from [09 - Web-only](09-foundry-iq-webiq-only.ipynb) to this **Web + Blob** lab,
then [11 - complete knowledge base](../python-notebooks/11-foundry-iq-knowledge-base.ipynb).
This notebook is standalone: it does not reuse or change either neighboring lab's definitions.
It requires no SharePoint site, app credential, ACL index, APIM, broker or agent.

Search agentic retrieval, chat inference, embedding/vectorizer inference and Web IQ calls may incur charges.
Stored Web IQ authentication authorizes both portal and notebook retrieval; protect source-author permissions
because the source definition contains the profile key. Do not print the source PUT response or export that key.


Set the kernel working directory to `c-sharp-notebooks`. Configuration loads `..\.env` first, then `..\.env.azure`, both overriding process settings; set `WORKSHOP_PREFIX` before running (no prefix prompt).

Use Polyglot Notebooks / C# .NET Interactive. All helpers are inline; no console imports, prompts or infrastructure detection. Setup makes no HTTP requests; the final cell disposes the local client.


## Existing resources and configuration

Use an existing Search service and Foundry chat deployment, with the service-identity model grants above.
Complete [05 - Content Understanding](05-blob-content-understanding.ipynb) using the same prefix.
This lab uses its `<prefix>-cu-chunks` index of Content Understanding-enriched Blob documents.
This notebook references that index read-only: it does not upload files or create indexing resources.
The index's existing semantic configuration and managed-identity vectorizer remain unchanged.

Sign in with `az login` in `AZURE_TENANT_ID`. Set the notebook kernel's working directory to
`c-sharp-notebooks`. Configuration loads `..\.env` first and `..\.env.azure` second, both overriding
stale process settings. Required settings: `WORKSHOP_PREFIX`, `AZURE_TENANT_ID`, `AZURE_SEARCH_ENDPOINT`,
`AZURE_AI_ACCOUNT_NAME`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`, `WEBIQ_API_KEY`.
`AZURE_AI_MODEL_NAME` defaults to `gpt-5-mini`; it must describe the existing chat deployment.
Configuration selects `<prefix>-cu-chunks` automatically; no index-name change is needed.


In [1]:
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Net.Http;
using System.Threading;
using System.Threading.Tasks;
using System.Diagnostics;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Text.RegularExpressions;

public sealed class WebIqSettings(string tenant, string searchEndpoint, string account,
    string deployment, string model, string prefix, string webIqKey)
{
    public string Tenant { get; } = tenant;
    public string SearchEndpoint { get; } = searchEndpoint;
    public string Account { get; } = account;
    public string Deployment { get; } = deployment;
    public string Model { get; } = model;
    public string Prefix { get; } = prefix;
    public string WebIqKey { get; } = webIqKey;
    public string ModelEndpoint => $"https://{Account}.openai.azure.com";
    public string IndexName => Prefix + "-cu-chunks";
    public string BlobSourceName => Prefix + "-nb10-blob";
    public string SourceName => Prefix + "-nb10-webiq";
    public string BaseName => Prefix + "-nb10-kb";

    public static WebIqSettings Load(string root)
    {
        LoadEnvFile(Path.Combine(root, ".env"));
        LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Missing setting: " + name + ". See .env.example.");
        var settings = new WebIqSettings(Required("AZURE_TENANT_ID"), Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'),
            Required("AZURE_AI_ACCOUNT_NAME"), Required("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
            Environment.GetEnvironmentVariable("AZURE_AI_MODEL_NAME") ?? "gpt-5-mini",
            Required("WORKSHOP_PREFIX"), Required("WEBIQ_API_KEY"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Regex.IsMatch(Prefix, @"^[a-z][a-z0-9]{2,19}$"))
            throw new InvalidOperationException("Use a tenant GUID and 3-20 lowercase letters/digits prefix starting with a letter.");
        if (!Uri.TryCreate(SearchEndpoint, UriKind.Absolute, out var endpoint) || endpoint.Scheme != "https" ||
            endpoint.UserInfo != "" || !endpoint.IsDefaultPort || endpoint.AbsolutePath != "/" ||
            endpoint.Query != "" || endpoint.Fragment != "" ||
            !Regex.IsMatch(endpoint.Host, @"^[a-z0-9][a-z0-9-]*\.search\.windows\.net$", RegexOptions.IgnoreCase))
            throw new InvalidOperationException("Use a public Azure HTTPS Search service root without credentials, path or query.");
        if (!Regex.IsMatch(Account, @"^[a-zA-Z0-9][a-zA-Z0-9-]*$") ||
            string.IsNullOrWhiteSpace(Deployment) || string.IsNullOrWhiteSpace(Model) || string.IsNullOrWhiteSpace(WebIqKey))
            throw new InvalidOperationException("Use a Foundry account name, chat deployment/model and non-empty WEBIQ_API_KEY.");
    }

    public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path)) return;
        foreach (string line in File.ReadAllLines(path))
        {
            string setting = line.Trim();
            if (setting.Length == 0 || setting.StartsWith('#')) continue;
            if (setting.StartsWith("export ", StringComparison.Ordinal)) setting = setting[7..].TrimStart();
            int equals = setting.IndexOf('=');
            if (equals <= 0) continue;
            string name = setting[..equals].Trim();
            string value = setting[(equals + 1)..].Trim();
            if (value.Length >= 2 && ((value[0] == '"' && value[^1] == '"') || (value[0] == '\'' && value[^1] == '\'')))
                value = value[1..^1];
            else
            {
                int comment = value.IndexOf(" #", StringComparison.Ordinal);
                if (comment >= 0) value = value[..comment].TrimEnd();
            }
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}

public sealed class WebIqTransport(HttpClient client, string tenant, Func<Task<string>>? tokenProvider = null)
{
    public const string Scope = "https://search.azure.com/.default";
    public const string ApiVersion = "2026-08-01-preview";
    readonly Func<Task<string>> getToken = tokenProvider ?? (() => AzureCliTokenAsync(tenant));

    public async Task<JsonObject> SendAsync(WebIqSettings settings, HttpMethod method, string path,
        JsonObject? body = null, bool allowMissing = false)
    {
        settings.Validate();
        string token;
        try { token = await getToken(); }
        catch (Exception error)
        {
            throw new InvalidOperationException(WebIqResults.Redact(error.Message, settings.WebIqKey));
        }
        using var request = new HttpRequestMessage(method, $"{settings.SearchEndpoint}/{path}?api-version={ApiVersion}");
        request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", token);
        if (body is not null)
            request.Content = new StringContent(body.ToJsonString(), Encoding.UTF8, "application/json");
        try
        {
            using var response = await client.SendAsync(request);
            string text = await response.Content.ReadAsStringAsync();
            int status = (int)response.StatusCode;
            string Safe(string value) => WebIqResults.Redact(value, settings.WebIqKey, token);
            Console.WriteLine($"HTTP status: {status}");
            Console.WriteLine("Request ID: " + Safe(response.Headers.TryGetValues("x-ms-request-id", out var ids)
                ? string.Join(", ", ids) : response.Headers.TryGetValues("request-id", out ids) ? string.Join(", ", ids) : ""));
            if (allowMissing && status == 404) return new JsonObject();
            if (!response.IsSuccessStatusCode)
            {
                Console.Error.WriteLine($"HTTP {status}: Search request failed. UTC (Date): {response.Headers.Date:R}");
                Console.Error.WriteLine(status switch
                {
                    401 => "Check Azure CLI sign-in and tenant; this is Search authorization, not proof of Web IQ key failure.",
                    403 => "Check Search RBAC/network access (index reads need a data role); model/vectorizer errors need Search managed-identity inference access.",
                    429 => "Quota or throttling; inspect Retry-After and service/profile capacity. No automatic billable retry.",
                    404 when path.StartsWith("indexes/") => "Complete lab 05 Content Understanding with the same prefix; do not create an index here.",
                    _ => "Inspect source activity and request identifiers; auth_missing_token concerns the stored Web IQ profile header."
                });
                foreach (var header in response.Headers)
                    if (header.Key.Contains("request", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Contains("correlation", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Contains("trace", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Equals("retry-after", StringComparison.OrdinalIgnoreCase))
                        Console.Error.WriteLine(header.Key + ": " + Safe(string.Join(", ", header.Value)));
                string safe = Safe(text);
                Console.Error.WriteLine("Response body: " + safe[..Math.Min(8000, safe.Length)]);
                throw new HttpRequestException($"HTTP {status}: Search request failed; inspect redacted diagnostics above.", null, response.StatusCode);
            }
            JsonObject result;
            try
            {
                result = string.IsNullOrWhiteSpace(text) && method != HttpMethod.Post ? new JsonObject()
                    : JsonNode.Parse(Safe(text))?.AsObject() ?? throw new JsonException();
            }
            catch (Exception error) when (error is JsonException or InvalidOperationException)
            {
                throw new InvalidOperationException("Search returned invalid JSON; no complete result available.");
            }
            if (path.EndsWith("/retrieve", StringComparison.Ordinal))
                WebIqResults.Print(result, status, settings.WebIqKey, token);
            else if (status == 206)
                throw new InvalidOperationException("Search returned a partial response; no complete result available.");
            return result;
        }
        catch (HttpRequestException error)
        {
            throw new HttpRequestException(WebIqResults.Redact(error.Message, settings.WebIqKey, token), null, error.StatusCode);
        }
        catch (OperationCanceledException)
        {
            throw new TimeoutException("Search request timed out before a complete response; inspect network/service health. No automatic retry.");
        }
    }

    public static async Task<string> AzureCliTokenAsync(string tenant)
    {
        if (!Guid.TryParseExact(tenant, "D", out _)) throw new InvalidOperationException("Invalid tenant GUID.");
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true
        };
        if (OperatingSystem.IsWindows())
            foreach (string argument in new[] { "/d", "/c", "az" }) start.ArgumentList.Add(argument);
        foreach (string argument in new[] { "account", "get-access-token", "--scope", Scope, "--tenant", tenant, "--output", "json" })
            start.ArgumentList.Add(argument);
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync();
        Task<string> errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited) process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out before Search/Web IQ. Run az login --tenant AZURE_TENANT_ID.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed before Search/Web IQ. Run az login --tenant AZURE_TENANT_ID.");
        try
        {
            return JsonNode.Parse(await output)?["accessToken"]?.GetValue<string>() is { Length: > 0 } token
                ? token : throw new JsonException();
        }
        catch (JsonException)
        {
            throw new InvalidOperationException("Azure CLI returned no usable access token.");
        }
    }
}

public static class WebIqResults
{
    public static string Redact(string text, params string[] secrets)
    {
        foreach (string secret in secrets.Where(secret => !string.IsNullOrEmpty(secret)))
        {
            text = text.Replace(secret, "<redacted>", StringComparison.Ordinal);
            string escaped = JsonSerializer.Serialize(secret)[1..^1];
            text = text.Replace(escaped, "<redacted>", StringComparison.Ordinal);
            text = text.Replace(Uri.EscapeDataString(secret), "<redacted>", StringComparison.Ordinal);
        }
        return text;
    }

    public static void ValidateQuery(string question, string outputMode)
    {
        if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
        if (outputMode is not ("answerSynthesis" or "extractiveData"))
            throw new ArgumentException("Use answerSynthesis or extractiveData.");
    }

    public static void Print(JsonObject result, int status, params string[] secrets)
    {
        void Write(string value) => Console.WriteLine(Redact(value, secrets));
        Console.WriteLine("\nActivity:");
        Write((result["activity"] ?? new JsonArray()).ToJsonString(new JsonSerializerOptions { WriteIndented = true }));
        Console.WriteLine("\nAnswer or extractive output:");
        foreach (JsonNode? message in result["response"] as JsonArray ?? new JsonArray())
            foreach (JsonNode? part in message?["content"] as JsonArray ?? new JsonArray())
                Write(part?["text"]?.ToString() ?? "");
        Console.WriteLine("\nReferences and retrieved passages:");
        foreach (JsonNode? reference in result["references"] as JsonArray ?? new JsonArray())
        {
            var data = reference?["sourceData"]?.DeepClone() as JsonObject ?? new JsonObject();
            try
            {
                if (JsonNode.Parse(data["content"]?.ToString() ?? "{}") is JsonObject parsed)
                    foreach (var property in parsed) data[property.Key] = property.Value?.DeepClone();
            }
            catch (JsonException) { }
            Write($"- {reference?["id"]} {data["title"]} {data["url"] ?? data["source/url"] ?? data["source"]?["url"]}");
            string content = (data["chunk"] ?? data["content"])?.ToString() ?? "";
            string safeContent = Redact(content, secrets);
            Write(safeContent[..Math.Min(2000, safeContent.Length)]);
        }
        if (status == 206 || result["activity"] is JsonArray activity &&
            activity.Any(step => step?["error"] is JsonNode error && error.ToJsonString() is not ("null" or "{}" or "\"\"" or "false")))
            throw new InvalidOperationException("Retrieval was partial or a source failed. Inspect the activity above.");
        if (result["references"] is not JsonArray references || references.Count == 0)
            Console.WriteLine("No references returned. Inspect activity; this alone does not establish an authentication failure.");
    }
}

WebIqSettings settings = WebIqSettings.Load(Path.GetFullPath(".."));
HttpClient client = new HttpClient(new HttpClientHandler { AllowAutoRedirect = false })
{
    Timeout = TimeSpan.FromSeconds(120)
};
Func<Task<string>> tokenProvider = () => WebIqTransport.AzureCliTokenAsync(settings.Tenant);
WebIqTransport transport = new WebIqTransport(client, settings.Tenant, () => tokenProvider());
Console.WriteLine("Search: " + settings.SearchEndpoint);
Console.WriteLine("Existing Blob index: " + settings.IndexName);
Console.WriteLine("Blob source: " + settings.BlobSourceName);
Console.WriteLine("Web IQ source: " + settings.SourceName);
Console.WriteLine("Knowledge base: " + settings.BaseName);


Search: https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net
Existing Blob index: dib01-cu-chunks
Blob source: dib01-nb10-blob
Web IQ source: dib01-nb10-webiq
Knowledge base: dib01-nb10-kb


## 1. Verify the existing Blob index

The read-only GET checks retrievable chunk/source fields and `chunk-semantic`; a count-only
query then verifies nonzero chunks visible to your Search identity. Missing schema or empty/inaccessible
data is a prerequisite failure, not an invitation to rebuild the index here.


In [2]:
JsonObject index = await transport.SendAsync(settings, HttpMethod.Get, $"indexes/{settings.IndexName}");
var fields = new Dictionary<string, JsonNode>();
foreach (JsonNode? field in index["fields"] as JsonArray ?? new JsonArray())
{
    if (field?["name"] is not JsonValue name) continue;
    fields[name.ToString()] = field;
    foreach (JsonNode? child in field["fields"] as JsonArray ?? new JsonArray())
        if (child?["name"] is JsonValue childName) fields[$"{name}/{childName}"] = child;
}
string[] required = ["chunk_id", "chunk", "title", "source/file_name", "source/url"];
if (required.Any(name => !fields.TryGetValue(name, out var field) ||
    field["retrievable"]?.ToString() == "false") ||
    fields["chunk"]["searchable"]?.ToString() != "true")
    throw new InvalidOperationException($"{settings.IndexName} does not have the expected retrievable content/source fields. Complete lab 05.");
if (!(index["semantic"]?["configurations"] as JsonArray ?? new JsonArray())
    .Any(config => config?["name"]?.ToString() == "chunk-semantic"))
    throw new InvalidOperationException($"{settings.IndexName} is missing semantic configuration chunk-semantic. Complete lab 05.");
JsonObject countResult = await transport.SendAsync(settings, HttpMethod.Post, $"indexes/{settings.IndexName}/docs/search",
    new JsonObject { ["search"] = "*", ["count"] = true, ["top"] = 0 });
if (countResult["@odata.count"] is not JsonValue countValue ||
    !countValue.TryGetValue<long>(out long count) || count <= 0)
    throw new InvalidOperationException($"No chunks are visible in {settings.IndexName}. Check ingestion and document access in lab 05.");
Console.WriteLine($"{settings.IndexName}: {count} visible chunks; existing schema unchanged.");


HTTP status: 200
Request ID: 4d6ad9fd-0015-4783-bd9d-e9fee6c52c2d
HTTP status: 200
Request ID: 5fc79e67-c102-4e21-9b2d-9c655ec27705
dib01-cu-chunks: 5 visible chunks; existing schema unchanged.


## 2. Create exactly two knowledge sources

The Blob `searchIndex` source points to the existing chunk index and its source metadata.
The Web IQ `mcpServer` source calls `https://api.microsoft.ai/v3/mcp` with a stored `x-apikey`
header and only the basic `web` tool allowlisted. No dynamic caller token is sent to Web IQ.
Every Web IQ call must request `contentFormat: passage`; do not send internal document text to Web IQ.
These PUTs create source definitions only, not indexes, indexers, skillsets or data sources.


In [3]:
JsonObject blobSource = new JsonObject()
    {
        ["name"] = settings.BlobSourceName,
        ["kind"] = "searchIndex",
        ["description"] = "Travel reimbursement and onboarding policy documents from the existing Blob index.",
        ["searchIndexParameters"] = new JsonObject
        {
            ["searchIndexName"] = settings.IndexName,
            ["semanticConfigurationName"] = "chunk-semantic",
            ["searchFields"] = new JsonArray(new JsonObject { ["name"] = "chunk" }),
            ["sourceDataFields"] = new JsonArray(
                new JsonObject { ["name"] = "chunk_id" }, new JsonObject { ["name"] = "chunk" },
                new JsonObject { ["name"] = "title" }, new JsonObject { ["name"] = "source/file_name" },
                new JsonObject { ["name"] = "source/url" })
        }
    };
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgesources/{settings.BlobSourceName}", blobSource);
Console.WriteLine($"Blob knowledge source ready: {settings.BlobSourceName} -> {settings.IndexName}");


HTTP status: 201
Request ID: 2c83bac6-fe79-4d40-a4a6-6122308c0c09
Blob knowledge source ready: dib01-nb10-blob -> dib01-cu-chunks


In [4]:
JsonObject webiqSource = new JsonObject()
    {
        ["name"] = settings.SourceName,
        ["kind"] = "mcpServer",
        ["description"] = "Basic Web IQ web search for public information. For every web call, explicitly set contentFormat to passage; use query-dependent passages, not text, html or markdown.",
        ["resultsProcessing"] = "rerank",
        ["mcpServerParameters"] = new JsonObject
        {
            ["serverURL"] = "https://api.microsoft.ai/v3/mcp",
            ["authentication"] = new JsonObject
            {
                ["kind"] = "storedHeaders",
                ["storedHeadersParameters"] = new JsonObject
                {
                    ["headers"] = new JsonObject { ["x-apikey"] = settings.WebIqKey }
                }
            },
            ["tools"] = new JsonArray(new JsonObject
            {
                ["name"] = "web",
                ["outputParsing"] = new JsonObject { ["kind"] = "auto" },
                ["maxOutputTokens"] = 4000
            })
        }
    };
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgesources/{settings.SourceName}", webiqSource);
Console.WriteLine($"Web IQ source ready: {settings.SourceName} | authentication: storedHeaders | tool: web");


HTTP status: 201
Request ID: 66d6bd44-5dc1-4164-8409-2424c248cccb
Web IQ source ready: dib01-nb10-webiq | authentication: storedHeaders | tool: web


## 3. Create the independent Web + Blob knowledge base

The existing Foundry chat deployment handles query planning and answer synthesis via Search's
managed identity; no model API key is embedded. `low` reasoning effort keeps the initial comparison
small. The KB contains exactly the two sources above. Its routing instructions distinguish internal
policy evidence from public web evidence without disclosing internal passages to Web IQ.


In [5]:
JsonObject knowledgeBase = new JsonObject()
    {
        ["name"] = settings.BaseName,
        ["description"] = "Existing Blob policies plus live public web evidence from Web IQ.",
        ["retrievalInstructions"] = $"Use {settings.BlobSourceName} for internal travel reimbursement and onboarding policies. Use {settings.SourceName} for current public information and external facts. For comparisons use both sources. Never send internal document text to Web IQ. For every web call explicitly set contentFormat to passage. Use specific public search terms, locations and dates from the question.",
        ["answerInstructions"] = "Answer concisely and cite supporting reference IDs. Distinguish internal policy facts from public web facts. If the evidence is missing, say you do not know.",
        ["outputMode"] = "answerSynthesis",
        ["knowledgeSources"] = new JsonArray(new JsonObject { ["name"] = settings.BlobSourceName },
            new JsonObject { ["name"] = settings.SourceName }),
        ["models"] = new JsonArray(new JsonObject
        {
            ["kind"] = "azureOpenAI",
            ["azureOpenAIParameters"] = new JsonObject
            {
                ["resourceUri"] = settings.ModelEndpoint,
                ["deploymentId"] = settings.Deployment,
                ["modelName"] = settings.Model
            }
        }),
        ["retrievalReasoningEffort"] = new JsonObject { ["kind"] = "low" }
    };
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgebases/{settings.BaseName}", knowledgeBase);
Console.WriteLine("Web IQ + Blob knowledge base ready: " + settings.BaseName);


HTTP status: 201
Request ID: 7cc630f2-aa6e-4172-abef-6da3c53ffe5b
Web IQ + Blob knowledge base ready: dib01-nb10-kb


## 4. Retrieve Blob evidence, web evidence, then compare both

Each cell shows the actual retrieve request and its source-specific `knowledgeSourceParams`.
Start with `answerSynthesis` to see a grounded answer, then rerun the same question with
`output_mode = 'extractiveData'` to inspect retrieved evidence without final-answer synthesis.
Keep `low` KB reasoning effort for the first comparison; optionally change it to `medium` in the
KB PUT and compare activity, latency and evidence coverage. Extractive output can still require
chat inference for planning; it is not a guarantee of no model usage.

Inspect activity for the source actually queried, and references for the supporting passage and URL.
Mixed retrieval makes both sources available; the planner may choose only one if evidence is missing.
Do not infer that both were used from an answer alone, or treat an empty reference list as proof of
an authentication failure. HTTP 206 or any source activity error is rejected, even when an answer exists.


In [6]:
string outputMode = "answerSynthesis"; // Compare with "extractiveData" on the same questions.


In [7]:
string blobQuestion = $"Using {settings.BlobSourceName}, what receipts do I need for travel reimbursement?";
WebIqResults.ValidateQuery(blobQuestion, outputMode);
JsonObject blobRequest = new JsonObject
{
    ["messages"] = new JsonArray(new JsonObject
    {
        ["role"] = "user",
        ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = blobQuestion })
    }),
    ["outputMode"] = outputMode,
    ["includeActivity"] = true,
    ["maxRuntimeInSeconds"] = 60,
    ["knowledgeSourceParams"] = new JsonArray(new JsonObject
    {
        ["knowledgeSourceName"] = settings.BlobSourceName, ["kind"] = "searchIndex",
        ["includeReferences"] = true, ["includeReferenceSourceData"] = true
    })
};
JsonObject blobResult = await transport.SendAsync(settings, HttpMethod.Post,
    $"knowledgebases/{settings.BaseName}/retrieve", blobRequest);


HTTP status: 200
Request ID: 1a290163-fc37-484b-b380-1deec252a3ef

Activity:
[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "startedAt": "2026-11-16T04:37:24.1748255Z",
    "completedAt": "2026-11-16T04:37:25.3946706Z",
    "inputTokens": 2174,
    "outputTokens": 43,
    "elapsedMs": 1219,
    "model": {
      "modelName": "gpt-5-mini",
      "deploymentId": "workshop-model"
    }
  },
  {
    "type": "searchIndex",
    "id": 1,
    "startedAt": "2026-11-16T04:37:25.398812Z",
    "completedAt": "2026-11-16T04:37:25.6079213Z",
    "knowledgeSourceName": "dib01-nb10-blob",
    "queryTime": "2026-11-16T04:37:25.5399305Z",
    "count": 5,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "travel reimbursement receipts required travel reimbursement receipts list onboarding travel expense receipts company policy",
      "filter": null,
      "semanticConfigurationName": "chunk-semantic",
      "sourceDataFields": [
        {
          "name": "title"
        },
   

In [8]:
string webQuestion = "What are the current public IRS guidelines for business travel receipt records?";
WebIqResults.ValidateQuery(webQuestion, outputMode);
JsonObject webRequest = new JsonObject
{
    ["messages"] = new JsonArray(new JsonObject
    {
        ["role"] = "user",
        ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = webQuestion })
    }),
    ["outputMode"] = outputMode,
    ["includeActivity"] = true,
    ["maxRuntimeInSeconds"] = 60,
    ["knowledgeSourceParams"] = new JsonArray(new JsonObject
    {
        ["knowledgeSourceName"] = settings.SourceName, ["kind"] = "mcpServer",
        ["includeReferences"] = true, ["includeReferenceSourceData"] = true
    })
};
JsonObject webResult = await transport.SendAsync(settings, HttpMethod.Post,
    $"knowledgebases/{settings.BaseName}/retrieve", webRequest);


HTTP status: 200
Request ID: 601b1b7f-f4d2-43c5-afd2-04fca4266f46

Activity:
[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "startedAt": "2026-11-16T04:37:33.2521191Z",
    "completedAt": "2026-11-16T04:37:34.8053457Z",
    "inputTokens": 2169,
    "outputTokens": 77,
    "elapsedMs": 1552,
    "model": {
      "modelName": "gpt-5-mini",
      "deploymentId": "workshop-model"
    }
  },
  {
    "type": "mcpServer",
    "id": 1,
    "startedAt": "2026-11-16T04:37:34.8055487Z",
    "completedAt": "2026-11-16T04:37:35.0333745Z",
    "knowledgeSourceName": "dib01-nb10-webiq",
    "queryTime": "2026-11-16T04:37:34.8056279Z",
    "count": 4,
    "elapsedMs": 169,
    "mcpServerArguments": {
      "toolName": "web",
      "toolArguments": {
        "query": "IRS guidelines business travel receipt records what to keep receipts substantiation 2024 IRS travel expenses receipts records business travel publication 463",
        "maxResults": 10,
        "contentFormat": "passage"
      }

In [9]:
string mixedQuestion = $"Use {settings.BlobSourceName} for our internal travel receipt policy and {settings.SourceName} for public IRS travel receipt guidelines. Compare the requirements; do not send internal text to the web.";
WebIqResults.ValidateQuery(mixedQuestion, outputMode);
JsonObject mixedRequest = new JsonObject
{
    ["messages"] = new JsonArray(new JsonObject
    {
        ["role"] = "user",
        ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = mixedQuestion })
    }),
    ["outputMode"] = outputMode,
    ["includeActivity"] = true,
    ["maxRuntimeInSeconds"] = 60,
    ["knowledgeSourceParams"] = new JsonArray(new JsonObject
    {
        ["knowledgeSourceName"] = settings.BlobSourceName, ["kind"] = "searchIndex",
        ["includeReferences"] = true, ["includeReferenceSourceData"] = true
    },
    new JsonObject
    {
        ["knowledgeSourceName"] = settings.SourceName, ["kind"] = "mcpServer",
        ["includeReferences"] = true, ["includeReferenceSourceData"] = true
    })
};
JsonObject mixedResult = await transport.SendAsync(settings, HttpMethod.Post,
    $"knowledgebases/{settings.BaseName}/retrieve", mixedRequest);


HTTP status: 200
Request ID: fc43a7f8-2ea6-4b25-bc47-d606dccc2461

Activity:
[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "startedAt": "2026-11-16T04:37:45.6958582Z",
    "completedAt": "2026-11-16T04:37:47.0229418Z",
    "inputTokens": 2198,
    "outputTokens": 45,
    "elapsedMs": 1326,
    "model": {
      "modelName": "gpt-5-mini",
      "deploymentId": "workshop-model"
    }
  },
  {
    "type": "searchIndex",
    "id": 1,
    "startedAt": "2026-11-16T04:37:47.0231332Z",
    "completedAt": "2026-11-16T04:37:47.1292941Z",
    "knowledgeSourceName": "dib01-nb10-blob",
    "queryTime": "2026-11-16T04:37:47.0865807Z",
    "count": 5,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "travel receipt policy travel reimbursement receipts internal policy requirements receipt details onboarding travel receipts company policy",
      "filter": null,
      "semanticConfigurationName": "chunk-semantic",
      "sourceDataFields": [
        {
          "name": "title

## Diagnose evidence and authorization boundaries

HTTP failures include a request ID and redacted error body. Activity errors also fail the cell;
check `knowledgeSourceName`, counts and errors before calling the answer successful.
Web IQ `auth_missing_token` points to the stored profile header/authorization, not your Search token.
Search 403 on index reads indicates your Search data role; chat/vectorizer authorization failures
require checking Search's managed-identity model grant. HTTP 429 indicates quota or throttling.
An empty Content Understanding Blob index requires completing 05. Sparse web references may require a more precise
public query; inspect reference URLs and source passages rather than assuming an authentication error.

The portal can retrieve through this same KB using the stored Web IQ header. Protect source-author
access and never display the echoed Web IQ source definition, stored key or bearer token.
After comparing both output modes, continue to [11 - Web + Blob + SharePoint](../python-notebooks/11-foundry-iq-knowledge-base.ipynb)
when its separate SharePoint indexing and user-authorization prerequisites are ready.

HTTP diagnostics include UTC Date, request/correlation IDs and Retry-After when available. Client timeout is 120 seconds; CLI token acquisition is bounded to 60 seconds. No automatic paid retries or credential values are printed.


### Web IQ + Content Understanding Blob knowledge base in the portal

![Foundry IQ portal showing active Blob and Web IQ sources, a cited public Python-release answer and an internal reimbursement-policy answer](../docs/webiq-blob-knowledge-base-portal.png)

The left pane shows the two active knowledge sources: `<prefix>-nb10-blob`, which references lab 05's `<prefix>-cu-chunks` index, and `<prefix>-nb10-webiq`. The public Python-release question uses Web IQ references; the reimbursement-policy question demonstrates retrieval from the internal travel handbook. Inspect each answer's references and Debug activity to confirm which source supplied its evidence.

The screenshot is illustrative, not a current Python-version reference or a required answer. Your resource prefix, release dates, retrieved passages and responses can differ. These are two separate questions, not proof that one request combined both sources; use the notebook's mixed-question cell to test that. Sending questions in the portal makes additional live retrieval requests and can incur model and Web IQ charges.

## Optional cleanup

Keep `CLEAN_UP = False` to compare results later. Opting in deletes this KB first and then only
its two source definitions. It does not delete the existing Blob index, files or neighboring labs.


In [10]:
bool CLEAN_UP = false;
if (CLEAN_UP)
{
    foreach (var (collection, name) in new[]
    {
        ("knowledgebases", settings.BaseName), ("knowledgesources", settings.SourceName),
        ("knowledgesources", settings.BlobSourceName)
    })
    {
        await transport.SendAsync(settings, HttpMethod.Delete, $"{collection}/{name}", allowMissing: true);
        Console.WriteLine("Removed or already absent: " + name);
    }
}
else Console.WriteLine("Cleanup disabled; Web IQ + Blob resources remain available.");


Cleanup disabled; Web IQ + Blob resources remain available.


In [11]:
client.Dispose();


## References

- [MCP server knowledge sources and authentication](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Retrieve action and response](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)
- [Answer synthesis and per-request output mode](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-answer-synthesis)
